# 06: Turn measurement counts into a comparison

You will load a hardware result, turn counts into probabilities, plot them beside the ideal answer, and calculate a simple distance.
Every calculation is written in this notebook.
The default data comes from a real experiment on September 9, 2026 and is labeled `historical_replay`.

## 1. Choose a result file

Leave `use_live_result` off to work with the recorded example. Turn it on after notebook 05 saves your own completed result.

In [ ]:
from pathlib import Path

# Find the checkout whether Jupyter started in the repository or this folder.
ROOT = next(
    p
    for p in (Path.cwd(), *Path.cwd().parents)
    if (p / "pyproject.toml").is_file() and (p / "flagquantum").is_dir()
)
WORKSHOP = ROOT / "workshops/flagos2026"
OUTPUTS = WORKSHOP / "outputs"
OUTPUTS.mkdir(exist_ok=True)
print("Your result folder:", OUTPUTS)


In [ ]:
import json

use_live_result = False
if use_live_result:
    result_path = OUTPUTS / "quafu-bell.json"
else:
    result_path = WORKSHOP / "reference_results/quafu_bell_replay.json"
record = json.loads(result_path.read_text())
print("Source:", record["source"])
print("Task:", record["task_id"])
print("Counts:", record["counts"])


## 2. Check what was measured

The result must be completed and the counts must add up to the reported number of shots.
Missing outcomes are treated as zero in the next step.

In [ ]:
labels = ["00", "01", "10", "11"]
counts = record["counts"]
assert record["status"] == "completed"
assert all(
    key in labels and type(value) is int and value >= 0 for key, value in counts.items()
)
shots = sum(counts.values())
assert shots > 0 and shots == record["shots"]
print("Total shots:", shots)


## 3. Convert counts to relative frequencies

Divide each count by the total number of shots. For example, 443 occurrences out of 1024 shots gives a frequency of about 0.433.
The ideal Bell distribution is `[0.5, 0, 0, 0.5]`.

In [ ]:
observed = [counts.get(label, 0) / shots for label in labels]
ideal = [0.5, 0.0, 0.0, 0.5]
for label, expected, measured in zip(labels, ideal, observed):
    print(f"{label}: ideal={expected:.3f}, observed={measured:.3f}")


## 4. Plot the two distributions

Keep the source label and task ID on the chart so others know which experiment they are looking at.

In [ ]:
import matplotlib.pyplot as plt

x = list(range(len(labels)))
plt.bar([v - 0.2 for v in x], ideal, width=0.4, label="Ideal")
plt.bar([v + 0.2 for v in x], observed, width=0.4, label=record["source"])
plt.xticks(x, labels)
plt.ylabel("Probability / relative frequency")
plt.title("Task " + record["task_id"])
plt.legend()
plt.show()


## 5. Calculate how far apart they are

Take the absolute difference for each outcome, add those differences, and divide by two.
This is called *total variation distance*. Zero means the distributions match; larger values mean they differ more.

In [ ]:
differences = [abs(expected - measured) for expected, measured in zip(ideal, observed)]
distance = sum(differences) / 2
print("Differences:", differences)
print("Total variation distance:", distance)


## Discuss with your group

This compares measurements in the computational (Z) basis. It is not state fidelity, and these measurements alone cannot prove entanglement.
Noise and the randomness of a finite number of shots can both change measured frequencies.

What other measurements would help you learn more about the state?
If you repeated the experiment tomorrow, how could you distinguish ordinary sampling variation from a change in the device?
Try increasing the number of shots in a future, separately authorized hardware run and compare the uncertainty.